#Engine v1 — PyTuroChamp

**AI idea:** Hand-crafted evaluation · **Era:** 1948 · **Week 3** · **Source:** [github.com/stevexyz/PyTuroChamp](https://github.com/stevexyz/PyTuroChamp) ·



## 0 · Setup — the same cell in every engine notebook

Installs python-chess if needed, finds Stockfish, and defines the small toolkit every notebook uses: `stockfish()`, `show()`, `play_game()`, `play_match()`, `greedy_bot()`, `save_results()`, `save_pgn()`. If the last line says Stockfish was **not found**, paste its path into `STOCKFISH_PATH` and re-run.

In [ ]:
# ---------------------------------------------------------------- shared setup (same in every engine notebook)
#  THE TOOLKIT -- what this cell leaves behind. The same nine functions and four names are
#  in every engine notebook of the ladder, so what you learn here works in v2 through v8.
#  (In a cell, run  help(play_match)  to see any of these again without scrolling back.)
#
#  NAMES YOU CAN READ
#    IN_COLAB        True on Google Colab. When True this cell also apt-installs Stockfish.
#    STOCKFISH_PATH  Empty by default -- paste your own path here if the finder below fails.
#    STOCKFISH       Full path to the Stockfish binary, or None. Printed on the last line.
#    VALUES          Piece values in centipawns: pawn 100, knight 300, bishop 300,
#                    rook 500, queen 900, king 0 (the king is never captured, so it scores 0).
#
#  OPEN AN ENGINE, SEE A BOARD
#    stockfish(elo=None, threads=1)
#                    Start a Stockfish process and return it. Pass elo=1350 to weaken it to
#                    a club player. ALWAYS call .quit() on what it returns, or the process
#                    stays alive after the cell ends.
#    show(board, size=320)
#                    Draw the board in the notebook with the last move highlighted.
#                    Displays; returns nothing.
#
#  PLAY GAMES BETWEEN TWO BOTS
#    A "bot" here is any function that takes a board and returns a legal move --
#    greedy_bot, random_bot, and the turing_bot you write in Section 5 all qualify.
#    play_game(white_fn, black_fn, max_plies=160, start=None)
#                    Play one game; returns the finished board. start takes a FEN if you
#                    want to begin somewhere other than move 1.
#    play_match(fn_a, fn_b, games=10, max_plies=160, seed=1)
#                    Play a match with colours alternating; returns (points_for_a, decisive)
#                    where a win is 1 point, a draw 0.5, and decisive counts games that
#                    actually finished in a result. This is how you measure a change.
#
#  BOTS AND SCORING -- your opponents and your yardstick
#    random_bot(board)
#                    Plays a legal move at random. The floor: anything you write should beat it.
#    material(board, color)
#                    Centipawn material balance from color's point of view (+ is good for color).
#    greedy_bot(board)
#                    The Setup-Week baseline, three rules: play mate if you see it, else take
#                    the most material, else choose at random among equals. Scores about
#                    26-28 out of 30 against random_bot (it moves with the seed) --
#                    beat THIS and you have built something.
#
#  SAVE YOUR EVIDENCE -- both write into the portfolio layout when it exists, else here
#    save_pgn(board, name, white="?", black="?")
#                    Write a finished game to games/<name>; returns the path it used.
#    save_results(rows, header, engine_folder)
#                    Write engines/<engine_folder>/results.csv; returns the path it used.
#                    rows is a list of lists, header a list of column names.
# ----------------------------------------------------------------------------------------
%pip install chess -q
import sys, os, shutil, csv, time, random, platform, re
import chess, chess.engine, chess.pgn, chess.svg
from IPython.display import display, SVG, clear_output

import asyncio
import sys

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "-q", "stockfish"], capture_output=True)

STOCKFISH_PATH = "C:\\Users\\Kenny\\Documents\\University Of Toledo\\Fall 26\\MATH 4900\\stockfish\\stockfish-windows-x86-64-universal.exe"      # laptop users: paste the full path here if the finder below cannot see it
STOCKFISH = next((p for p in [STOCKFISH_PATH, os.environ.get("STOCKFISH_PATH"), shutil.which("stockfish"),
                              "/usr/games/stockfish", "/usr/local/bin/stockfish", "/opt/homebrew/bin/stockfish",
                              "./stockfish", "./stockfish.exe"] if p and os.path.isfile(p)), None)


def count_files(folder_path, extension):
    ''' 
    Count how many files of a given extension exist in a directory.
        Args: 
            folder_path: string
                Path string to the target directory
            extension: string
                Extension string to match (.csv, .pgn)
        Returns:
            Integer count of matching files
    '''
    if not os.path.exists(folder_path):
        return 0

    ext = extension.lower()
    if not ext.startswith('.'):
        ext = f'.{ext}'

    return len(
        [ # This is just list comprehension
            f
            for f in os.listdir(folder_path)
            if f.lower().endswith(ext)
            and os.path.isfile(os.path.join(folder_path, f))
        ]
    )
# end count_files

def stockfish(elo=None, threads=1):
    """Open a Stockfish process (optionally rating-limited). Remember to .quit() it."""
    if not STOCKFISH:
        raise FileNotFoundError("Stockfish not found -- set STOCKFISH_PATH (see Setup Week Kit, Step 5.1)")
    eng = chess.engine.SimpleEngine.popen_uci(STOCKFISH)
    if elo is not None:
        opt = eng.options["UCI_Elo"]
        eng.configure({"UCI_LimitStrength": True, "UCI_Elo": max(opt.min, min(opt.max, elo))})
    return eng

def show(board, size=320):
    """Draw `board` in the notebook, highlighting the last move. Displays; returns nothing."""
    display(SVG(chess.svg.board(board, size=size, lastmove=board.peek() if board.move_stack else None)))

def play_game(white_fn, black_fn, max_plies=160, start=None):
    """Two Python functions (board -> move) play a game. Returns the final board."""
    b = chess.Board(start) if start else chess.Board()
    while not b.is_game_over() and len(b.move_stack) < max_plies:
        b.push((white_fn if b.turn == chess.WHITE else black_fn)(b))
    return b


def get_next_match_number(folder_path):
    """Scan existing .pgn files in folder_path, find the max Match number, and return max + 1."""
    if not os.path.exists(folder_path):
        return 1

    pattern = re.compile(r"^Match(\d+)", re.IGNORECASE)
    match_nums = [
        int(pattern.search(f).group(1))
        for f in os.listdir(folder_path)
        if f.lower().endswith(".pgn") and pattern.search(f)
    ]

    return max(match_nums) + 1 if match_nums else 1

def play_match(fn_a, fn_b, games=10, max_plies=160, seed=1, save_games=True, save_result=True):
    """fn_a vs fn_b with colours alternating. Returns (points for A, decisive games)."""
    pts, decisive = 0.0, 0

    name_a = getattr(fn_a, "__name__", "Bot_A")
    name_b = getattr(fn_b, "__name__", "Bot_B")

    # Determine subfolder structure
    p1, p2 = sorted([str(name_a), str(name_b)])
    subfolder = f"{p1}_vs_{p2}"
    target_dir = os.path.join("..", "games", subfolder)

    if save_games:
        os.makedirs(target_dir, exist_ok=True)
        match_num = get_next_match_number(target_dir)

    # Determine padding width based on total game count (e.g., 01-10 vs 001-100)
    pad_width = len(str(games))

    for i in range(games):
        random.seed(seed + i)
        a_white = i % 2 == 0

        white_player = name_a if a_white else name_b
        black_player = name_b if a_white else name_a

        b = (
            play_game(fn_a, fn_b, max_plies)
            if a_white
            else play_game(fn_b, fn_a, max_plies)
        )
        res = b.result() if b.is_game_over() else "1/2-1/2"

        if res != "1/2-1/2" and b.is_game_over():
            decisive += 1
        if res == "1-0":
            pts += 1 if a_white else 0
        elif res == "0-1":
            pts += 0 if a_white else 1
        else:
            pts += 0.5

        if save_games:
            # Construct a per-game indexed filename: e.g., "round1_game_01"
            game_num = str(i + 1).zfill(pad_width)
            game_filename = f"Match{match_num:02d}_Game{game_num}.pgn"

            save_pgn(
                board=b,
                name=game_filename,
                white=white_player,
                black=black_player,
                target_dir=target_dir,
            )

    if save_result:
        save_results(
            rows = [[subfolder, 20, pts, decisive]],
            white = white_player,
            black = black_player
            )

    return pts, decisive

'''
def play_match(fn_a, fn_b, games=10, max_plies=160, seed=1, save_games = False, **kwargs):
    """fn_a vs fn_b with colours alternating. Returns (points for A, decisive games)."""
    pts, decisive = 0.0, 0
    for i in range(games):
        random.seed(seed + i)
        a_white = (i % 2 == 0)
        b = play_game(fn_a, fn_b, max_plies) if a_white else play_game(fn_b, fn_a, max_plies)
        res = b.result() if b.is_game_over() else "1/2-1/2"
        if res != "1/2-1/2" and b.is_game_over(): decisive += 1
        if res == "1-0":   pts += 1 if a_white else 0
        elif res == "0-1": pts += 0 if a_white else 1
        else:              pts += 0.5
        
    return pts, decisive
'''

def random_bot(board):
    """A legal move chosen at random. The weakest possible opponent -- your baseline's baseline."""
    return random.choice(list(board.legal_moves))

VALUES = {chess.PAWN: 100, chess.KNIGHT: 300, chess.BISHOP: 300, chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}
def material(board, color):
    """Material balance in centipawns from `color`'s point of view: positive means `color` is ahead."""
    return sum(VALUES[p.piece_type] * (1 if p.color == color else -1) for p in board.piece_map().values())

def greedy_bot(board):
    """Mate if you see it, else grab the most material, random tie-break."""
    me, best, moves = board.turn, None, []
    for m in board.legal_moves:
        board.push(m); s = 10**6 if board.is_checkmate() else material(board, me); board.pop()
        if best is None or s > best: best, moves = s, [m]
        elif s == best: moves.append(m)
    return random.choice(moves)

def save_pgn(board, name, target_dir, white="?", black="?"):
    """Save an individual game as PGN into ../games/<subfolder>/."""
    
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "Gambit Lab"
    game.headers["White"] = str(white)
    game.headers["Black"] = str(black)
    game.headers["Result"] = board.result() if board.is_game_over() else "*"

    file_path = os.path.join(target_dir, name)
    with open(file_path, "w", encoding="utf-8") as f:
        print(game, file=f, end="\n\n")

    print(f"saved game -> {file_path}")
    return file_path

'''
def save_pgn(board, name, white="?", black="?"):
    """Save a finished game as PGN into games/ (portfolio layout) if that folder exists, else here."""
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "Gambit Lab"; game.headers["White"] = white; game.headers["Black"] = black
    game.headers["Result"] = board.result() if board.is_game_over() else "*"
    path = os.path.join('..','games', name +'.pgn')
    with open(path, "w") as f: print(game, file=f, end="\n\n")
    print("saved game ->", path)
    return path
'''

def save_results(
    rows,
    header=["experiment", "games", "points_for_change", "decisive"],
    white="?",
    black="?",
    ):
    
    target_dir = os.path.join("..", "results")
    os.makedirs(target_dir, exist_ok=True)

    # Name the file by the matchup sorted alphabetically
    p1, p2 = sorted([str(white), str(black)])
    path = os.path.join(target_dir, f"{p1}_vs_{p2}.csv")

    # Check if file exists to determine if the header is needed
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(header)
        w.writerows(rows)

    print(f"Results appended to {path}")
    return path

'''
def save_results(rows, header, engine_folder, file_name):
    """Write results.csv into engines/<folder>/ if that folder exists (portfolio layout), else here."""
    folder = os.path.join('..','results')
    path = os.path.join(folder,file_name+'.csv')
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    print(f"saved {len(rows)} rows -> {path}")
    return path
'''

def make_eval_bot(eval_fn, name=None):
    """Factory that creates a 1-ply bot function using any eval_fn(board, me)."""

    def bot(board):
        me = board.turn
        best = None
        moves = []

        for m in board.legal_moves:
            board.push(m)
            s = 10**6 if board.is_checkmate() else eval_fn(board, me)
            board.pop()

            if best is None or s > best:
                best, moves = s, [m]
            elif s == best:
                moves.append(m)

        return random.choice(moves)

    # Set __name__ so play_match and folder/file saving can read it automatically
    bot.__name__ = name if name else eval_fn.__name__.replace("eval", "bot")
    return bot

print("python-chess", chess.__version__, "| Stockfish:", STOCKFISH or "NOT FOUND (only the engine cells need it)")

Note: you may need to restart the kernel to use updated packages.
python-chess 1.11.2 | Stockfish: C:\Users\Kenny\Documents\University Of Toledo\Fall 26\MATH 4900\stockfish\stockfish-windows-x86-64-universal.exe


C:\Users\Kenny\AppData\Local\Temp\ipykernel_52260\3862532604.py:60: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\Kenny\AppData\Local\Temp\ipykernel_52260\3862532604.py:60: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


## 1 · Read first (20 minutes)

An <strong>evaluation function</strong> is a recipe that turns a position into a number: plus if White is better, minus if Black is. Turing's recipe counted material as a <em>ratio</em> (your material divided by the opponent's), then added small bonuses for mobility (how many moves you have), safety of the king, castling, and giving check.

PyTuroChamp is a modern Python reconstruction of that 1948 'paper machine'. It reads and plays through the same UCI/XBoard interface Stockfish uses. This week you build a Turing-style evaluator yourself first, then run the real thing.

**The shape of it:** <code>evaluate(position)</code> → a number &nbsp;·&nbsp; try every legal move once &nbsp;·&nbsp; keep the best number

### Why this idea, and why now

Every engine needs some way to say 'this position is good for me'. The 1948 answer was: write the judgement down as arithmetic. Material dominates (a queen is worth about nine pawns), but Turing added terms a strong player feels — pieces that can move are worth more than pieces that cannot; a king that can be checked is a liability; castling is worth a little.

Look-ahead is minimal: one ply for quiet moves, a bit more for captures ('considerable moves' — the seed of what later engines call quiescence). It plays badly by modern standards, and that is the point: you will feel exactly which weaknesses the next seven engines were invented to fix.

### Reading list — in this order

1. [Turochamp — Chess Programming Wiki](https://www.chessprogramming.org/Turochamp) — The history and the actual evaluation rules Turing used; ten minutes.
2. [Claude Shannon — Chess Programming Wiki](https://www.chessprogramming.org/Claude_Shannon) — Links to the paper itself and to the ideas it seeded.
3. [PyTuroChamp README](https://github.com/stevexyz/PyTuroChamp) — How the reconstruction is organised: pyturochamp.py (Turing's heuristics), bare.py, newt.py, ptc_xboard.py.

## 2 · Run the engine

Open the repository in a browser: [https://github.com/stevexyz/PyTuroChamp](https://github.com/stevexyz/PyTuroChamp).



Get it from GitHub
```bash
git clone https://github.com/stevexyz/PyTuroChamp.git
cd PyTuroChamp
pip install chess
```

In [ ]:
# Optional: run the clone from here (handy on Colab, where there is no terminal). Set CLONE = True and run.
CLONE = True
if CLONE:
    import subprocess
    for cmd in [
    "git clone https://github.com/stevexyz/PyTuroChamp.git",
    ]:
        print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-600:])
else:
    print("CLONE is False -- run the commands in a terminal, or set CLONE = True here.")

$ git clone https://github.com/stevexyz/PyTuroChamp.git
Cloning into 'PyTuroChamp'...





**Terminal — play it through the harness**

```bash
python scripts/first_engine_game.py --engine "python PyTuroChamp/ptc_xboard.py"
```

**Python — or open it directly**

```python
import chess.engine
engine = chess.engine.SimpleEngine.popen_uci(["python", "PyTuroChamp/ptc_xboard.py"])
```

Task: Open PyTuroChamp, plays a full game against an opponent you choose, prints every move as it is played, then shows the final board and writes the game to games/ as a PGN.

Write a cell for each opponent:

1. Against you
2. Against Stockfish

## 3 · Make Changes and Measure

Write an evaluation function turing_eval (material ratio, mobility, a check penalty, castling rights) and engine function turing_bot (try every legal move once, keep the best, random tie-break) by hand. Every cell runs without the real engine installed.

Turing's recipe, as arithmetic. Material is a <em>ratio</em> (his choice — it makes trades matter more when you are ahead), then three small bonuses: mobility, king safety, and castling rights. Everything is measured from the point of view of the side that just moved, so the bot can compare its options.

In [3]:
def turing_eval(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score = 100 * (mine + 100) / (theirs + 100)          # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 20
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score

In [51]:
turing_bot = make_eval_bot(turing_eval,"Turing_Bot")

b = chess.Board()
print("Turing-bot's first move:", b.san(turing_bot(b)))
pts, dec = play_match(turing_bot, greedy_bot, games=10, max_plies=120)
print(f"Turing-bot vs Greedy: {pts}/10 points, {dec} decisive games")

Turing-bot's first move: e3
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game01.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game02.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game03.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game04.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game05.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game06.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game07.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game08.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game09.pgn
saved game -> ..\games\Turing_Bot_vs_greedy_bot\Match01_Game10.pgn
Results appended to ..\results\Turing_Bot_vs_greedy_bot.csv
Turing-bot vs Greedy: 4.5/10 points, 3 decisive games


Make at least one change to the evaluation function, while keeping everything else identical. For example:

1. Changing the the mobility weight 0.5 → 1.0

2. Material difference instead of ratio

3. The check penalty 20 → 50; the castling bonus 5 → 0.

Test run 20 games to see whether the engine gets stronger or weaker against the original and save the results in results.csv.

Write in RECIPE.md: what you changed and why that one; the results pasted; one sentence interpreting them that claims no more than twenty games can support.


In [52]:

def turing_eval_mobile(board, me):
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)

    score = 100 * (mine + 100) / (theirs + 100)

    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 1.0 * (my_moves - their_moves)               # <-- THE ONE CHANGE (was 0.5)
    if board.turn == me and board.is_check(): score -= 20
    score += 5 if board.has_castling_rights(me) else 0
    return score

turing_bot_mobile = make_eval_bot(turing_eval_mobile,'Turing_Bot_Mobile')


pts, dec = play_match(turing_bot_mobile, turing_bot, games=20, max_plies=120)
print(f"mobility x2 vs original: {pts}/20 ({dec} decisive)")


saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game01.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game02.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game03.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game04.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game05.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game06.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game07.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game08.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game09.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game10.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game11.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game12.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Mobile\Match01_Game13.pgn
saved game -> ..\games\Turing_Bot_vs_T

In [ ]:
sf_eng = stockfish(elo=1350)

def sf_bot(board):
    return sf_eng.play(board, chess.engine.Limit(time=0.1)).move

pts, dec = play_match(sf_bot, turing_bot, games = 20, max_plies=120)
print(f"mobility x2 vs original: {pts}/20 ({dec} decisive)")

sf_eng.quit()

saved game -> ..\games\sf_bot_vs_turing_bot\01_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\02_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\03_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\04_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\05_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\06_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\07_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\08_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\09_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\10_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\11_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\12_game_sf_bot_vs_turing_bot.pgn
saved game -> ..\games\sf_bot_vs_turing_bot\13_game_sf_bot_vs_turing_bot.pgn

(19.5, 19)

In [53]:
import numpy as np

def turing_eval_mine(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score = np.sqrt(mine) * np.sqrt(theirs) #100 * (mine + 100) / (theirs + 100)          # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 20
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score

sqrt_bot = make_eval_bot(turing_eval_mine,'Sqrt_Bot')
play_match(sqrt_bot,turing_bot,games=20,max_plies=120)

saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game01.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game02.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game03.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game04.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game05.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game06.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game07.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game08.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game09.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game10.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game11.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game12.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game13.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game14.pgn
saved game -> ..\games\Sqrt_Bot_vs_Turing_Bot\Match01_Game15.pgn
saved game -> ..\games\Sq

(3.5, 13)

In [63]:
def turing_eval_must_lose(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score = -( 100 * (mine + 100) / (theirs + 100) )         # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 200
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score

must_lose = make_eval_bot(turing_eval_must_lose,'Must_Lose')

In [64]:
play_match(must_lose,random_bot)

saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game01.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game02.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game03.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game04.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game05.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game06.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game07.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game08.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game09.pgn
saved game -> ..\games\Must_Lose_vs_random_bot\Match02_Game10.pgn
Results appended to ..\results\Must_Lose_vs_random_bot.csv


(6.0, 2)

In [65]:
play_match(must_lose,turing_bot)

saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game01.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game02.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game03.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game04.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game05.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game06.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game07.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game08.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game09.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match01_Game10.pgn
Results appended to ..\results\Must_Lose_vs_Turing_Bot.csv


(2.0, 6)